# Modelování nelineárních křivek poptávky v maloobchodě pomocí PROC GAM


## Shrnutí pro vedení

Tento notebook používá **PROC GAM** k modelování týdenních prodejů
maloobchodních jednotek jako hladké, nelineární funkce ceny na regále, teploty
v obchodě (proxy pro sezónnost) a výdajů na propagační akce, s parametrickým
efektem indikátoru akce. Zobecněné aditivní modely umožňují kategorijnímu
manažerovi odhalit skutečné zakřivené tvary cenové elasticity a sezónní
poptávky, které by lineární regrese přehlédla, což podporuje přesnější
rozhodování o cenách a akcích.


## Zdroje dat

| Dataset | Řádky | Granularita | Klíčové proměnné | Popis |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | týden | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Syntetická týdenní řada prodejních dat pro jeden vlajkový obchod v průběhu 100 po sobě jdoucích týdnů (téměř dva sezónní cykly). Vygenerována přímo v programu pomocí `call streaminit(20250531)` a `rand()`. Týdenní prodeje jednotek se řídí Poissonovým procesem poptávky, jehož střední hodnotu určuje exponenciální křivka cenové odezvy, kvadratický efekt teploty/sezónnosti vrcholící kolem 72F a konkávní (odmocninový) nárůst z výdajů na akci, plus diskrétní indikátor akčního týdne. |


# Modelování nelineárních křivek poptávky v maloobchodě pomocí PROC GAM

Poptávka v maloobchodě jen zřídka reaguje na cenu, počasí nebo výdaje na akce
přímočaře. Malé snížení ceny běžného zboží může objem sotva pohnout, zatímco
překročení psychologické cenové hranice může vyvolat prudký skok; poptávka
ovlivněná počasím vrcholí v příjemném středním rozsahu a klesá na obou
extrémech; a nárůst z akcí vykazuje klesající výnosy s rostoucími výdaji.

**PROC GAM** (zobecněné aditivní modely) proloží každý faktor vlastním hladkým
splajnovým členem, takže tvar každé křivky poptávky určují data — nikoli
rigidní lineární předpoklad. Zde modelujeme týdenní prodeje jednotek pro jeden
vlajkový maloobchod v průběhu 100 po sobě jdoucích týdnů, kombinujeme
parametrický indikátor akce s vyhlazovacími splajny na ceně, teplotě a
výdajích na akci za předpokladu Poissonovy odezvy.


## Krok 1 — Vygenerování syntetické týdenní prodejní řady

Simulujeme 100 po sobě jdoucích týdnů (téměř dva sezónní cykly) dat z místa
prodeje pro jeden vlajkový obchod. Proces generování dat je záměrně
nelineární, abychom mohli potvrdit, že GAM odhalí realistické tvary:

- **Price** (cena) řídí objem prostřednictvím exponenciální křivky odezvy
  (`exp(-1.1 * Price)`), takže poptávka prudce roste s klesající cenou.
- **Temperature** (teplota) slouží jako proxy pro sezónnost s kvadratickým
  vrcholem kolem 72F, modelující návštěvnost řízenou komfortem.
- **PromoSpend** (výdaje na akci) přináší konkávní, odmocninový nárůst
  (klesající výnosy).
- Diskrétní indikátor **Promotion** (akce) přidává parametrický skokový efekt
  v akčních týdnech.

Týdenní `Units` (prodané kusy) jsou generovány z Poissonova rozdělení, což
odpovídá povaze počtu prodaných kusů.


In [1]:
data store_sales;
   CALL streaminit(20250531);
   DÉLKA Promotion $3;
   OPAKUJ Week = 1 TO 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      KDYŽ rand("uniform") < 0.28 PAK OPAKUJ;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      KONEC;
      JINAK OPAKUJ;
         Promotion  = "No";
         PromoSpend = 0;
      KONEC;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      VÝSTUP;
   KONEC;
SPUSTIT;



NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Krok 2 — Profil simulovaných dat

Rychlá `PROC MEANS` potvrzuje, že proměnné pokrývají rozumné maloobchodní
rozsahy před modelováním: počty jednotek jsou nezáporná celá čísla, cena se
seskupuje kolem několika dolarů, teplota prochází celým sezónním cyklem a
výdaje na akci jsou nulové v neakčních týdnech.


In [2]:
PROCEDURA PRŮMĚRY data=store_sales n mean MIN MAX maxdec=2;
   PROMĚNNÁ Units Price Temperature PromoSpend;
   ŠTÍTEK Units="Prodané kusy" Price="Cena" Temperature="Teplota"
         PromoSpend="Výdaje na akci";
SPUSTIT;


                                                  The MEANS Procedure

 Variable     Label                   N           Mean     Minimum     Maximum
 -----------------------------------------------------------------------------
 Units        Prodané kusy          100           7.03        0.00      103.00
 Price        Cena                  100           3.15        2.81        3.48
 Temperature  Teplota               100          55.50       22.72       83.49
 PromoSpend   Výdaje na akci        100         128.76        0.00      779.00
 -----------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 3 — Proložení plného aditivního modelu poptávky

Základní model kombinuje:

- `param(Promotion)` — parametrický (lineární) efekt indikátoru akčního
  týdne, deklarovaný ve statementu `CLASS`.
- `spline(Price, df=4)` — kubický vyhlazovací splajn zachycující zakřivenou
  cenovou odezvu.
- `spline(Temperature, df=4)` — hladká křivka sezónnosti.
- `spline(PromoSpend, df=3)` — nárůst z akcí s klesajícími výnosy.

Protože prodeje jednotek jsou počty, zadáváme `dist=poisson` (log spojení).
`method=gcv` nechává zobecněnou křížovou validaci řídit hladkost jakékoli
složky bez explicitního přepsání stupňů volnosti. Statement `OUTPUT` zapisuje
predikce a rezidua za pozorování do `gam_fit`.

Procedura hlásí **Deviance 43,59** oproti **Null Deviance 2041,12** — čtyři
hladké a parametrický faktor dohromady vysvětlují téměř veškerou variabilitu,
kterou by ponechal model pouze s konstantou — a **AIC 254,61**. Parametrický
odhad `PROMOTIONYES` je kladný (+0,41 na logaritmické škále), což potvrzuje
nárůst objemu z akcí, a splajn ceny nese silně záporný lineární trend
(−1,71), typický pro klesající poptávku.


In [3]:
PROCEDURA gam data=store_sales PLOTS=components(CLM commonaxes);
   TŘÍDA Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   VÝSTUP out=gam_fit predicted residual;
   ŠTÍTEK Units="Prodané kusy" Promotion="Akce" Price="Cena"
         Temperature="Teplota" PromoSpend="Výdaje na akci";
SPUSTIT;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     Prodané kusy
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Cena)                     4.0000    


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Krok 4 — Zaměřený model ceny a sezónnosti

Pro štíhlejší cenový přehled model přeproložíme jen se dvěma nejvíce
rozhodovacně relevantními hladkými faktory — cenou a teplotou — a dáme ceně
navíc flexibilitu (`df=5`), aby vyřešila případný zlom poblíž psychologické
cenové hranice. Indikátor akce zůstává zachován jako parametrická kontrola.

Vynechání výdajů na akci zvyšuje **Deviance na 62,80** a **AIC na 269,82**,
obojí vyšší než 43,59 a 254,61 plného modelu. Parametrický člen
`PROMOTIONYES` zde také pohltí více akčního signálu (+1,73 oproti +0,41),
protože splajn výdajů už není přítomen, aby jej nesl. Splajn ceny si
zachovává svůj záporný trend (−1,51), takže základní příběh poptávky je mezi
specifikacemi stabilní.


In [4]:
PROCEDURA gam data=store_sales PLOTS=components(CLM);
   TŘÍDA Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   ŠTÍTEK Units="Prodané kusy" Promotion="Akce" Price="Cena"
         Temperature="Teplota";
SPUSTIT;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     Prodané kusy
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Cena)                     5.0000         5.0000
Spline(Teplota)                  4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretace výsledků

Tabulka **Regression Model Analysis** uvádí lineární trend v rámci každé
složky plus parametrický efekt akce. Kladný koeficient `PROMOTIONYES` (+0,41
v plném modelu, +1,73 ve štíhlejším) potvrzuje očekávaný nárůst objemu z
akcí, zatímco záporný lineární trend na splajnu ceny (−1,71 a −1,51) odráží
klasickou klesající poptávku. Malý kladný lineární člen splajnu teploty
(+0,03) je očekávaný: jeho skutečný příběh je zakřivení kolem komfortního
vrcholu 72F, které jediný lineární koeficient nedokáže shrnout.

Tabulka **Smoothing Model Analysis** uvádí stupně volnosti, které každý
splajn spotřebuje. Cena a teplota čerpají po 4 efektivních DF (5 pro cenu ve
štíhlejším modelu) a výdaje na akci 3 — vše výrazně nad jediným DF, které by
použila přímka, což je přesně důvod, proč by lineární regrese tyto zakřivené
vztahy přehlédla.

Tabulka **Fit Statistics** umožňuje obě specifikace přímo porovnat. Plný
model se čtyřmi faktory dosahuje Deviance 43,59 a AIC 254,61 oproti 62,80 a
269,82 štíhlejšího modelu; obě kritéria upřednostňují plný model, což
ukazuje, že výdaje na akci a indikátor akce přidávají vysvětlující sílu nad
rámec ceny a teploty samotné. Vzhledem k Null Deviance 2041,12 oba modely
zachycují naprostou většinu variability poptávky.

Společně tyto tabulky dávají kategorijnímu manažerovi kvantifikovaný,
datově podložený příběh poptávky: strmá cenová odezva, která informuje
hloubku slevy, sezónní teplotní okno a nárůst z akcí s klesajícími výnosy —
mnohem ostřejší vodítko než jediný odhad lineární elasticity. (PROC GAM
také přijímá `plots=components` k vykreslení křivek dílčích predikcí pro
každý hladký člen; číselné tabulky složek výše jsou zdrojem, ze kterého jsou
tyto křivky vykresleny.)
